In [ ]:
from rdkit.Chem import PandasTools
import numpy as np
import pandas as pd
from rdkit import DataStructs
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
import random
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import LeaveOneOut
from sklearn import preprocessing
# from genetic_selection import GeneticSelectionCV
from mordred import Calculator, descriptors

## Descriptor helpers
Functions to turn reaction SMILES into 3D Mordred feature rows.


In [ ]:
def calculate_descriptors_for_molecule(smiles, prefix=""):
    """
    Compute Mordred descriptors for one SMILES string.
    Returns a descriptor dict, or None on failure.
    """
    if pd.isna(smiles):
        return None
        
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
            
        mol_3d = Chem.AddHs(mol)
        Chem.EmbedMolecule(mol_3d, randomSeed=0xf006d)
        
        try:
            Chem.MMFFOptimizeMolecule(mol_3d)
        except:
            pass
            
        calc = Calculator(descriptors)
        desc_dict = calc(mol_3d)
        
        numeric_descriptors = {}
        for key, value in desc_dict.items():
            try:
                float_value = float(value)
                numeric_descriptors[f"{prefix}{key}"] = float_value
            except (ValueError, TypeError):
                continue
                
        return numeric_descriptors
        
    except Exception as e:
        print(f"Error for SMILES {smiles}: {e}")
        return None

In [ ]:
def process_reactions(data):
    """
    Process all reactions and return descriptor rows.
    """
    all_descriptors = []
    
    for idx, reaction_smiles in enumerate(data['SMILES']):
        if pd.isna(reaction_smiles):
            all_descriptors.append({})
            continue
        
        print(f"Processing reaction {idx+1}/{len(data)}: {reaction_smiles}")
        
        reaction_descriptors = {}
        
        if '>>' in str(reaction_smiles):
            reagents_part, products_part = reaction_smiles.split('>>')
            
            reagents = [r.strip() for r in reagents_part.split('.') if r.strip()]
            for i, reagent in enumerate(reagents):
                prefix = f"reagent{i+1}_"
                desc_dict = calculate_descriptors_for_molecule(reagent, prefix)
                if desc_dict:
                    reaction_descriptors.update(desc_dict)
            
            products = [p.strip() for p in products_part.split('.') if p.strip()]
            for i, product in enumerate(products):
                prefix = f"product{i+1}_"
                desc_dict = calculate_descriptors_for_molecule(product, prefix)
                if desc_dict:
                    reaction_descriptors.update(desc_dict)
                    
        # else:
        #     prefix_reag = "reagent1_"
        #     desc_dict_reag = calculate_descriptors_for_molecule(reaction_smiles, prefix_reag)
        #     if desc_dict_reag:
        #         reaction_descriptors.update(desc_dict_reag)
            
        #     prefix_prod = "product1_"
        #     desc_dict_prod = calculate_descriptors_for_molecule(reaction_smiles, prefix_prod)
        #     if desc_dict_prod:
        #         reaction_descriptors.update(desc_dict_prod)
        
        all_descriptors.append(reaction_descriptors)
    
    return all_descriptors

## Load curated reactions
Read `dfsmile.csv` (reaction keys + SMILES) and align with descriptor generation.


In [ ]:
data = pd.read_csv("../data/dfsmile.csv", index_col=False)
if 'Unnamed: 0' in data.columns:
    data.drop('Unnamed: 0', axis=1, inplace=True)
print(f"Загружено реакций: {len(data)}")

In [ ]:
data

In [ ]:
# data.drop([17, 16], inplace=True)
# data.reset_index(drop=True)

In [ ]:
print("Computing descriptors...")
all_descriptors_list = process_reactions(data)

descriptors_df = pd.DataFrame(all_descriptors_list)

result_df = pd.concat([data, descriptors_df], axis=1)

print("Processing finished.")
print(f"Итоговый размер: {result_df.shape}")

result_df.to_csv('reactions_with_descriptors_fixed.csv', index=False)
print("Saved to 'reactions_with_descriptors_fixed.csv'")

desc_columns = [col for col in result_df.columns if any(x in col for x in ['reagent', 'product'])]
print(f"Колонок с дескрипторами: {len(desc_columns)}")
print(f"Заполненных значений: {result_df[desc_columns].count().sum()}")
print(f"Процент заполнения: {result_df[desc_columns].count().sum() / (len(result_df) * len(desc_columns)) * 100:.2f}%")

## Merge descriptors with targets
Run the reaction loop, attach labels, and preview the wide table.


In [ ]:
result_df

## Descriptor column census
Identify descriptor blocks (`reagent*`, `product*`) and basic coverage stats.


In [ ]:
desc_columns = [col for col in result_df.columns if any(x in col for x in ['reagent', 'product'])]
empty_cols = result_df[desc_columns].columns[result_df[desc_columns].isna().all()].tolist()

print(f"Полностью пустых столбцов: {len(empty_cols)}")
cleaned_df = result_df.drop(columns=empty_cols)
print(f"Осталось столбцов: {len(cleaned_df.columns)}")

## Column pruning / QC
Optional drops and sanity checks before modelling exports.


In [ ]:
initial_columns = len(result_df.columns)
print(f"Исходное количество столбцов: {initial_columns}")

cleaned_df = result_df.dropna(axis=1)
final_columns = len(cleaned_df.columns)
print(f"Количество столбцов после удаления: {final_columns}")
print(f"Удалено столбцов: {initial_columns - final_columns}")
print(f"Сохранено: {final_columns} столбцов")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def complete_pca_analysis_with_parity(df, target_column, n_components_range=None):
    """
    Полный PCA анализ для малого набора данных (69 rows)
    """
    if n_components_range is None:
        n_components_range = [10, 20, 30, 40, 60]
    
    print("=" * 70)
    print("FULL PCA ANALYSIS (69 ROWS)")
    print("=" * 70)
    
    print("\n1. DATA PREPARATION")
    print("-" * 40)
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    desc_columns = [col for col in X.columns if any(x in col for x in ['reagent', 'product'])]
    X_desc = X[desc_columns]
    
    print(f"Всего rows: {X_desc.shape[0]}")
    print(f"Всего features: {X_desc.shape[1]}")
    print(f"Target variable: {target_column}")
    
    print("\n2. FEATURE SCALING")
    print("-" * 40)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_desc)
    
    print("Scaling finished")
    print(f"Среднее после масштабирования: {np.mean(X_scaled):.2f}")
    print(f"Стандартное отклонение: {np.std(X_scaled):.2f}")

    print("\n3. PCA EXPLAINED VARIANCE")
    print("-" * 40)
    
    pca_full = PCA().fit(X_scaled)
    explained_variance_full = np.cumsum(pca_full.explained_variance_ratio_)
    
    n_80 = np.argmax(explained_variance_full >= 0.80) + 1
    n_90 = np.argmax(explained_variance_full >= 0.90) + 1
    n_95 = np.argmax(explained_variance_full >= 0.95) + 1
    
    print(f"Компонент для 80% variance: {n_80}")
    print(f"Компонент для 90% variance: {n_90}")
    print(f"Компонент для 95% variance: {n_95}")
    
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(explained_variance_full) + 1), explained_variance_full, 'b-', linewidth=2)
    plt.axhline(y=0.80, color='r', linestyle='--', alpha=0.7, label='80% variance')
    plt.axhline(y=0.90, color='g', linestyle='--', alpha=0.7, label='90% variance')
    plt.axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95% variance')
    plt.axvline(x=n_80, color='r', linestyle=':', alpha=0.5)
    plt.axvline(x=n_90, color='g', linestyle=':', alpha=0.5)
    plt.axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)
    plt.xlabel('Number of components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Cumulative explained PCA variance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 2)
    plt.bar(range(1, 21), pca_full.explained_variance_ratio_[:20])
    plt.xlabel('PCA component index')
    plt.ylabel('Explained variance')
    plt.title('Explained variance by component (first 20)')
    plt.grid(True, alpha=0.3)
    
    print("\n4. СРАВНЕНИЕ РАЗЛИЧНЫХ DIFFERENT NUMBERS OF PCA COMPONENTS")
    print("-" * 50)
    
    results = {}
    models = {
        'Ridge': Ridge(alpha=1.0, random_state=42),
        'DecisionTree': DecisionTreeRegressor(max_depth=3, min_samples_split=10, random_state=42),
        'RandomForest': RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42),
        'ExtraTrees': ExtraTreesRegressor(
                        n_estimators=100,
                        max_depth=3,
                        min_samples_split=10,
                        min_samples_leaf=5,
                        max_features=0.5,
                        bootstrap=True,
                        random_state=42)
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    
    for n_comp in n_components_range:
        print(f"\nPCA с {n_comp} componentsами:")
        print("-" * 30)
        
        n_comp_actual = min(n_comp, X_scaled.shape[1])
        
        pca = PCA(n_components=n_comp_actual, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        
        explained_variance = np.sum(pca.explained_variance_ratio_)
        print(f"Explained variance: {explained_variance:.3f}")
        
        results[n_comp] = {
            'pca': pca,
            'X_pca': X_pca,
            'explained_variance': explained_variance,
            'model_scores': {}
        }
        
        for model_name, model in models.items():
            try:
                cv_scores = cross_val_score(model, X_pca, y, cv=kf, scoring='r2')
                mean_score = np.mean(cv_scores)
                std_score = np.std(cv_scores)
                
                results[n_comp]['model_scores'][model_name] = {
                    'mean_r2': mean_score,
                    'std_r2': std_score
                }
                
                print(f"  {model_name:15} | R²: {mean_score:.3f} ± {std_score:.3f}")
                
            except Exception as e:
                print(f"  {model_name:15} | Error: {e}")
                results[n_comp]['model_scores'][model_name] = {
                    'mean_r2': np.nan,
                    'std_r2': np.nan
                }

    print("\n5. SEARCHING BEST CONFIGURATION")
    print("-" * 40)
    
    best_score = -np.inf
    best_config = None
    
    for n_comp in n_components_range:
        for model_name in models.keys():
            if (model_name in results[n_comp]['model_scores'] and 
                not np.isnan(results[n_comp]['model_scores'][model_name]['mean_r2'])):
                score = results[n_comp]['model_scores'][model_name]['mean_r2']
                if score > best_score:
                    best_score = score
                    best_config = (n_comp, model_name)
    
    if best_config is None:
        raise ValueError("Не удалось найти рабочую конфигурацию моделей")
    
    best_n_comp, best_model_name = best_config
    print(f"ЛУЧШАЯ КОНФИГУРАЦИЯ: {best_n_comp} components, модель: {best_model_name}")
    print(f"ЛУЧШИЙ R² SCORE: {best_score:.3f}")

    plt.subplot(2, 2, 3)
    colors = ['blue', 'green', 'red', "pink"]
    for idx, model_name in enumerate(models.keys()):
        scores = []
        valid_n_components = []
        
        for n_comp in n_components_range:
            if (model_name in results[n_comp]['model_scores'] and 
                not np.isnan(results[n_comp]['model_scores'][model_name]['mean_r2'])):
                scores.append(results[n_comp]['model_scores'][model_name]['mean_r2'])
                valid_n_components.append(n_comp)
        
        if scores:
            plt.plot(valid_n_components, scores, 
                    marker='o', linewidth=2, label=model_name, color=colors[idx % len(colors)])
    
    plt.xlabel('Number of PCA components')
    plt.ylabel('R² score')
    plt.title('Model quality vs number of components')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 4)
    explained_variances = [results[n_comp]['explained_variance'] for n_comp in n_components_range]
    plt.plot(n_components_range, explained_variances, 'purple', marker='s', linewidth=2)
    plt.xlabel('Number of PCA components')
    plt.ylabel('Explained variance')
    plt.title('Explained variance vs components')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n6. АНАЛИЗ TARGET AND PARITY PLOTS")
    print("-" * 40)
    
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 3, 1)
    plt.hist(y, bins=15, alpha=0.7, edgecolor='black')
    plt.xlabel('Target value')
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {target_column}')
    plt.grid(True, alpha=0.3)
    
    y_stats = {
        'mean': np.mean(y),
        'std': np.std(y),
        'min': np.min(y),
        'max': np.max(y),
        'range': np.max(y) - np.min(y)
    }
    
    print(f"Статистика целевой переменной '{target_column}':")
    print(f"  Mean: {y_stats['mean']:.3f}")
    print(f"  Стандартное отклонение: {y_stats['std']:.3f}")
    print(f"  Range: [{y_stats['min']:.3f}, {y_stats['max']:.3f}]")
    print(f"  Размах: {y_stats['range']:.3f}")
    
    plt.subplot(2, 3, 2)
    plt.boxplot(y)
    plt.ylabel('Value')
    plt.title(f'Boxplot {target_column}')
    plt.grid(True, alpha=0.3)
    
    Q1 = np.percentile(y, 25)
    Q3 = np.percentile(y, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = y[(y < lower_bound) | (y > upper_bound)]
    print(f"Выбросов в целевой переменной: {len(outliers)}")
    
    print(f"\n7. PARITY PLOTS ДЛЯ ЛУЧШЕЙ КОНФИГУРАЦИИ")
    print("-" * 50)
    
    best_pca = PCA(n_components=min(best_n_comp, X_scaled.shape[1]), random_state=42)
    X_pca_best = best_pca.fit_transform(X_scaled)
    
    best_model = models[best_model_name]
    best_model.fit(X_pca_best, y)
    y_pred = best_model.predict(X_pca_best)
    
    r2_parity = r2_score(y, y_pred)
    rmse_parity = np.sqrt(mean_squared_error(y, y_pred))
    mae_parity = mean_absolute_error(y, y_pred)
    
    print(f"Лучшая конфигурация: {best_n_comp} components, {best_model_name}")
    print(f"Parity plot метрики:")
    print(f"  R²: {r2_parity:.3f}")
    print(f"  RMSE: {rmse_parity:.3f}")
    print(f"  MAE: {mae_parity:.3f}")
    
    plt.subplot(2, 3, 3)
    plt.scatter(y, y_pred, alpha=0.6, s=50)
    
    min_val = min(y.min(), y_pred.min())
    max_val = max(y.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal prediction')
    
    plt.xlabel('True values')
    plt.ylabel('Predicted values')
    plt.title(f'Parity Plot: {best_model_name}\nR² = {r2_parity:.3f}, RMSE = {rmse_parity:.3f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 4)
    residuals = y - y_pred
    plt.scatter(y_pred, residuals, alpha=0.6, s=50)
    plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
    plt.xlabel('Predicted values')
    plt.ylabel('Residuals')
    plt.title('Residual distribution')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 5)
    plt.hist(residuals, bins=15, alpha=0.7, edgecolor='black')
    plt.xlabel('Residuals')
    plt.ylabel('Frequency')
    plt.title('Residual histogram')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 6)
    model_names = []
    r2_scores = []
    
    for model_name in models.keys():
        if (model_name in results[best_n_comp]['model_scores'] and 
            not np.isnan(results[best_n_comp]['model_scores'][model_name]['mean_r2'])):
            model_names.append(model_name)
            r2_scores.append(results[best_n_comp]['model_scores'][model_name]['mean_r2'])
    
    if model_names:
        bars = plt.bar(model_names, r2_scores, color=plt.cm.Set3(np.linspace(0, 1, len(model_names))))
        plt.ylabel('R² score')
        plt.title(f'Model comparison ({best_n_comp} components)')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        for bar, score in zip(bars, r2_scores):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{score:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n8. АНАЛИЗ ОСТАТКОIn ДЛЯ {best_model_name}")
    print("-" * 50)
    
    residual_stats = {
        'mean': np.mean(residuals),
        'std': np.std(residuals),
        'min': np.min(residuals),
        'max': np.max(residuals),
        'rms': np.sqrt(np.mean(residuals**2))
    }
    
    print(f"Статистика остатков:")
    print(f"  Mean: {residual_stats['mean']:.3f} (should be ~0)")
    print(f"  Стандартное отклонение: {residual_stats['std']:.3f}")
    print(f"  Range: [{residual_stats['min']:.3f}, {residual_stats['max']:.3f}]")
    print(f"  RMS: {residual_stats['rms']:.3f}")
    
    if len(y_pred) > 1:
        correlation_residuals_pred = np.corrcoef(y_pred, residuals)[0, 1]
        print(f"Корреляция предсказаний и остатков: {correlation_residuals_pred:.3f}")
        print(f"  (should be near 0 for homoscedasticity)")
    else:
        print("Not enough data for correlation analysis")
    
    return {
        'scaler': scaler,
        'pca': best_pca,
        'model': best_model,
        'X_pca': X_pca_best,
        'results': results,
        'best_config': best_config,
        'final_metrics': {'r2': r2_parity, 'rmse': rmse_parity, 'mae': mae_parity},
        'target_stats': y_stats,
        'residual_stats': residual_stats,
        'y_true': y,
        'y_pred': y_pred,
        'residuals': residuals
    }

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def complete_pca_analysis_with_parity2(df, target_column, n_components_range=None):
    """
    Полный PCA анализ для малого набора данных (69 rows) ДЛЯ ДАННЫХ БОЛЬШЕЙ РАЗМЕРНОСТИ
    """
    if n_components_range is None:
        n_components_range = [10, 20, 30, 40, 60]
    
    print("=" * 70)
    print("FULL PCA ANALYSIS (69 ROWS)")
    print("=" * 70)
    
    print("\n1. DATA PREPARATION")
    print("-" * 40)
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    desc_columns = [col for col in X.columns if any(x in col for x in ['reagent', 'product'])]
    X_desc = X[desc_columns]
    
    print(f"Всего rows: {X_desc.shape[0]}")
    print(f"Всего features: {X_desc.shape[1]}")
    print(f"Target variable: {target_column}")
    
    print("\n2. FEATURE SCALING")
    print("-" * 40)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_desc)
    
    print("Scaling finished")
    print(f"Среднее после масштабирования: {np.mean(X_scaled):.2f}")
    print(f"Стандартное отклонение: {np.std(X_scaled):.2f}")

    print("\n3. PCA EXPLAINED VARIANCE")
    print("-" * 40)
    
    pca_full = PCA().fit(X_scaled)
    explained_variance_full = np.cumsum(pca_full.explained_variance_ratio_)
    
    n_80 = np.argmax(explained_variance_full >= 0.80) + 1
    n_90 = np.argmax(explained_variance_full >= 0.90) + 1
    n_95 = np.argmax(explained_variance_full >= 0.95) + 1
    
    print(f"Компонент для 80% variance: {n_80}")
    print(f"Компонент для 90% variance: {n_90}")
    print(f"Компонент для 95% variance: {n_95}")
    
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(explained_variance_full) + 1), explained_variance_full, 'b-', linewidth=2)
    plt.axhline(y=0.80, color='r', linestyle='--', alpha=0.7, label='80% variance')
    plt.axhline(y=0.90, color='g', linestyle='--', alpha=0.7, label='90% variance')
    plt.axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95% variance')
    plt.axvline(x=n_80, color='r', linestyle=':', alpha=0.5)
    plt.axvline(x=n_90, color='g', linestyle=':', alpha=0.5)
    plt.axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)
    plt.xlabel('Number of components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Cumulative explained PCA variance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 2)
    plt.bar(range(1, 21), pca_full.explained_variance_ratio_[:20])
    plt.xlabel('PCA component index')
    plt.ylabel('Explained variance')
    plt.title('Explained variance by component (first 20)')
    plt.grid(True, alpha=0.3)
    
    print("\n4. СРАВНЕНИЕ РАЗЛИЧНЫХ DIFFERENT NUMBERS OF PCA COMPONENTS")
    print("-" * 50)
    
    results = {}
    models = {
        'Ridge': Ridge(alpha=5.0, random_state=42),
        
        'DecisionTree': DecisionTreeRegressor(
            max_depth=4,
            min_samples_split=15,
            min_samples_leaf=8,
            max_features=0.3,
            random_state=42
        ),
        
        'RandomForest': RandomForestRegressor(
            n_estimators=100,
            max_depth=4,
            min_samples_split=15,
            min_samples_leaf=8,       
            max_features=0.3,
            bootstrap=True,
            random_state=42
        ),
        
        'ExtraTrees': ExtraTreesRegressor(
            n_estimators=150,
            max_depth=4,
            min_samples_split=15,
            min_samples_leaf=8,
            max_features=0.25,
            bootstrap=True,
            random_state=42
        )
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    
    for n_comp in n_components_range:
        print(f"\nPCA с {n_comp} componentsами:")
        print("-" * 30)
        
        n_comp_actual = min(n_comp, X_scaled.shape[1])
        
        pca = PCA(n_components=n_comp_actual, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        
        explained_variance = np.sum(pca.explained_variance_ratio_)
        print(f"Explained variance: {explained_variance:.3f}")
        
        results[n_comp] = {
            'pca': pca,
            'X_pca': X_pca,
            'explained_variance': explained_variance,
            'model_scores': {}
        }
        
        for model_name, model in models.items():
            try:
                cv_scores = cross_val_score(model, X_pca, y, cv=kf, scoring='r2')
                mean_score = np.mean(cv_scores)
                std_score = np.std(cv_scores)
                
                results[n_comp]['model_scores'][model_name] = {
                    'mean_r2': mean_score,
                    'std_r2': std_score
                }
                
                print(f"  {model_name:15} | R²: {mean_score:.3f} ± {std_score:.3f}")
                
            except Exception as e:
                print(f"  {model_name:15} | Error: {e}")
                results[n_comp]['model_scores'][model_name] = {
                    'mean_r2': np.nan,
                    'std_r2': np.nan
                }

    print("\n5. SEARCHING BEST CONFIGURATION")
    print("-" * 40)
    
    best_score = -np.inf
    best_config = None
    
    for n_comp in n_components_range:
        for model_name in models.keys():
            if (model_name in results[n_comp]['model_scores'] and 
                not np.isnan(results[n_comp]['model_scores'][model_name]['mean_r2'])):
                score = results[n_comp]['model_scores'][model_name]['mean_r2']
                if score > best_score:
                    best_score = score
                    best_config = (n_comp, model_name)
    
    if best_config is None:
        raise ValueError("Не удалось найти рабочую конфигурацию моделей")
    
    best_n_comp, best_model_name = best_config
    print(f"ЛУЧШАЯ КОНФИГУРАЦИЯ: {best_n_comp} components, модель: {best_model_name}")
    print(f"ЛУЧШИЙ R² SCORE: {best_score:.3f}")

    plt.subplot(2, 2, 3)
    colors = ['blue', 'green', 'red', "pink"]
    for idx, model_name in enumerate(models.keys()):
        scores = []
        valid_n_components = []
        
        for n_comp in n_components_range:
            if (model_name in results[n_comp]['model_scores'] and 
                not np.isnan(results[n_comp]['model_scores'][model_name]['mean_r2'])):
                scores.append(results[n_comp]['model_scores'][model_name]['mean_r2'])
                valid_n_components.append(n_comp)
        
        if scores:
            plt.plot(valid_n_components, scores, 
                    marker='o', linewidth=2, label=model_name, color=colors[idx % len(colors)])
    
    plt.xlabel('Number of PCA components')
    plt.ylabel('R² score')
    plt.title('Model quality vs number of components')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 4)
    explained_variances = [results[n_comp]['explained_variance'] for n_comp in n_components_range]
    plt.plot(n_components_range, explained_variances, 'purple', marker='s', linewidth=2)
    plt.xlabel('Number of PCA components')
    plt.ylabel('Explained variance')
    plt.title('Explained variance vs components')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n6. АНАЛИЗ TARGET AND PARITY PLOTS")
    print("-" * 40)
    
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 3, 1)
    plt.hist(y, bins=15, alpha=0.7, edgecolor='black')
    plt.xlabel('Target value')
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {target_column}')
    plt.grid(True, alpha=0.3)
    
    y_stats = {
        'mean': np.mean(y),
        'std': np.std(y),
        'min': np.min(y),
        'max': np.max(y),
        'range': np.max(y) - np.min(y)
    }
    
    print(f"Статистика целевой переменной '{target_column}':")
    print(f"  Mean: {y_stats['mean']:.3f}")
    print(f"  Стандартное отклонение: {y_stats['std']:.3f}")
    print(f"  Range: [{y_stats['min']:.3f}, {y_stats['max']:.3f}]")
    print(f"  Размах: {y_stats['range']:.3f}")
    
    plt.subplot(2, 3, 2)
    plt.boxplot(y)
    plt.ylabel('Value')
    plt.title(f'Boxplot {target_column}')
    plt.grid(True, alpha=0.3)
    
    Q1 = np.percentile(y, 25)
    Q3 = np.percentile(y, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = y[(y < lower_bound) | (y > upper_bound)]
    print(f"Выбросов в целевой переменной: {len(outliers)}")
    
    print(f"\n7. PARITY PLOTS ДЛЯ ЛУЧШЕЙ КОНФИГУРАЦИИ")
    print("-" * 50)
    
    best_pca = PCA(n_components=min(best_n_comp, X_scaled.shape[1]), random_state=42)
    X_pca_best = best_pca.fit_transform(X_scaled)
    
    best_model = models[best_model_name]
    best_model.fit(X_pca_best, y)
    y_pred = best_model.predict(X_pca_best)
    
    r2_parity = r2_score(y, y_pred)
    rmse_parity = np.sqrt(mean_squared_error(y, y_pred))
    mae_parity = mean_absolute_error(y, y_pred)
    
    print(f"Лучшая конфигурация: {best_n_comp} components, {best_model_name}")
    print(f"Parity plot метрики:")
    print(f"  R²: {r2_parity:.3f}")
    print(f"  RMSE: {rmse_parity:.3f}")
    print(f"  MAE: {mae_parity:.3f}")
    
    plt.subplot(2, 3, 3)
    plt.scatter(y, y_pred, alpha=0.6, s=50)
    
    min_val = min(y.min(), y_pred.min())
    max_val = max(y.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal prediction')
    
    plt.xlabel('True values')
    plt.ylabel('Predicted values')
    plt.title(f'Parity Plot: {best_model_name}\nR² = {r2_parity:.3f}, RMSE = {rmse_parity:.3f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 4)
    residuals = y - y_pred
    plt.scatter(y_pred, residuals, alpha=0.6, s=50)
    plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
    plt.xlabel('Predicted values')
    plt.ylabel('Residuals')
    plt.title('Residual distribution')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 5)
    plt.hist(residuals, bins=15, alpha=0.7, edgecolor='black')
    plt.xlabel('Residuals')
    plt.ylabel('Frequency')
    plt.title('Residual histogram')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 6)
    model_names = []
    r2_scores = []
    
    for model_name in models.keys():
        if (model_name in results[best_n_comp]['model_scores'] and 
            not np.isnan(results[best_n_comp]['model_scores'][model_name]['mean_r2'])):
            model_names.append(model_name)
            r2_scores.append(results[best_n_comp]['model_scores'][model_name]['mean_r2'])
    
    if model_names:
        bars = plt.bar(model_names, r2_scores, color=plt.cm.Set3(np.linspace(0, 1, len(model_names))))
        plt.ylabel('R² score')
        plt.title(f'Model comparison ({best_n_comp} components)')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        for bar, score in zip(bars, r2_scores):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{score:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n8. АНАЛИЗ ОСТАТКОIn ДЛЯ {best_model_name}")
    print("-" * 50)
    
    residual_stats = {
        'mean': np.mean(residuals),
        'std': np.std(residuals),
        'min': np.min(residuals),
        'max': np.max(residuals),
        'rms': np.sqrt(np.mean(residuals**2))
    }
    
    print(f"Статистика остатков:")
    print(f"  Mean: {residual_stats['mean']:.3f} (should be ~0)")
    print(f"  Стандартное отклонение: {residual_stats['std']:.3f}")
    print(f"  Range: [{residual_stats['min']:.3f}, {residual_stats['max']:.3f}]")
    print(f"  RMS: {residual_stats['rms']:.3f}")
    
    if len(y_pred) > 1:
        correlation_residuals_pred = np.corrcoef(y_pred, residuals)[0, 1]
        print(f"Корреляция предсказаний и остатков: {correlation_residuals_pred:.3f}")
        print(f"  (should be near 0 for homoscedasticity)")
    else:
        print("Not enough data for correlation analysis")
    
    return {
        'scaler': scaler,
        'pca': best_pca,
        'model': best_model,
        'X_pca': X_pca_best,
        'results': results,
        'best_config': best_config,
        'final_metrics': {'r2': r2_parity, 'rmse': rmse_parity, 'mae': mae_parity},
        'target_stats': y_stats,
        'residual_stats': residual_stats,
        'y_true': y,
        'y_pred': y_pred,
        'residuals': residuals
    }

In [ ]:
def complete_pca_analysis(df, target_column, n_components_range=None):
    """
    Полный PCA анализ для малого набора данных (69 rows)
    """
    if n_components_range is None:
        n_components_range = [10, 20, 30, 40, 60,]
    
    print("=" * 70)
    print("FULL PCA ANALYSIS (69 ROWS)")
    print("=" * 70)
    
    print("\n1. DATA PREPARATION")
    print("-" * 40)
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    desc_columns = [col for col in X.columns if any(x in col for x in ['reagent', 'product'])]
    X_desc = X[desc_columns]
    
    print(f"Всего rows: {X_desc.shape[0]}")
    print(f"Всего features: {X_desc.shape[1]}")
    print(f"Target variable: {target_column}")
    
    print("\n2. FEATURE SCALING")
    print("-" * 40)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_desc)
    
    print("Scaling finished")
    print(f"Среднее после масштабирования: {np.mean(X_scaled):.2f}")
    print(f"Стандартное отклонение: {np.std(X_scaled):.2f}")

    print("\n3. PCA EXPLAINED VARIANCE")
    print("-" * 40)
    
    pca_full = PCA().fit(X_scaled)
    explained_variance_full = np.cumsum(pca_full.explained_variance_ratio_)
    
    n_80 = np.argmax(explained_variance_full >= 0.80) + 1
    n_90 = np.argmax(explained_variance_full >= 0.90) + 1
    n_95 = np.argmax(explained_variance_full >= 0.95) + 1
    
    print(f"Компонент для 80% variance: {n_80}")
    print(f"Компонент для 90% variance: {n_90}")
    print(f"Компонент для 95% variance: {n_95}")
    
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(explained_variance_full) + 1), explained_variance_full, 'b-', linewidth=2)
    plt.axhline(y=0.80, color='r', linestyle='--', alpha=0.7, label='80% variance')
    plt.axhline(y=0.90, color='g', linestyle='--', alpha=0.7, label='90% variance')
    plt.axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95% variance')
    plt.axvline(x=n_80, color='r', linestyle=':', alpha=0.5)
    plt.axvline(x=n_90, color='g', linestyle=':', alpha=0.5)
    plt.axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)
    plt.xlabel('Number of components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Cumulative explained PCA variance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 2)
    plt.bar(range(1, 21), pca_full.explained_variance_ratio_[:20])
    plt.xlabel('PCA component index')
    plt.ylabel('Explained variance')
    plt.title('Explained variance by component (first 20)')
    plt.grid(True, alpha=0.3)
    
    print("\n4. СРАВНЕНИЕ РАЗЛИЧНЫХ DIFFERENT NUMBERS OF PCA COMPONENTS")
    print("-" * 50)
    
    results = {}
    models = {
        'Ridge': Ridge(alpha=1.0, random_state=42),
        'DecisionTree': DecisionTreeRegressor(max_depth=3, min_samples_split=10, random_state=42),
        'RandomForest': RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42),
        'ExtraTrees': ExtraTreesRegressor(
                        n_estimators=100,
                        max_depth=3,
                        min_samples_split=10,
                        min_samples_leaf=5,
                        max_features=0.5,
                        bootstrap=True,
                        random_state=42)
        }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    
    for n_comp in n_components_range:
        print(f"\nPCA с {n_comp} componentsами:")
        print("-" * 30)
        
        pca = PCA(n_components=n_comp, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        
        explained_variance = np.sum(pca.explained_variance_ratio_)
        print(f"Explained variance: {explained_variance:.3f}")
        
        results[n_comp] = {
            'pca': pca,
            'X_pca': X_pca,
            'explained_variance': explained_variance,
            'model_scores': {}
        }
        
        for model_name, model in models.items():
            try:
                cv_scores = cross_val_score(model, X_pca, y, cv=kf, scoring='r2', n_jobs=-1)
                mean_score = np.mean(cv_scores)
                std_score = np.std(cv_scores)
                
                results[n_comp]['model_scores'][model_name] = {
                    'mean_r2': mean_score,
                    'std_r2': std_score
                }
                
                print(f"  {model_name:15} | R²: {mean_score:.3f} ± {std_score:.3f}")
                
            except Exception as e:
                print(f"  {model_name:15} | Error: {e}")

    plt.subplot(2, 2, 3)
    colors = ['blue', 'green', 'red', "pink"]
    for idx, model_name in enumerate(models.keys()):
        scores = [results[n_comp]['model_scores'][model_name]['mean_r2'] 
                 for n_comp in n_components_range 
                 if model_name in results[n_comp]['model_scores']]
        
        if scores:
            plt.plot(n_components_range[:len(scores)], scores, 
                    marker='o', linewidth=2, label=model_name, color=colors[idx])
    
    plt.xlabel('Number of PCA components')
    plt.ylabel('R² score')
    plt.title('Model quality vs number of components')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 4)
    explained_variances = [results[n_comp]['explained_variance'] for n_comp in n_components_range]
    plt.plot(n_components_range, explained_variances, 'purple', marker='s', linewidth=2)
    plt.xlabel('Number of PCA components')
    plt.ylabel('Explained variance')
    plt.title('Explained variance vs components')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n5. SELECTING OPTIMAL CONFIGURATION")
    print("-" * 40)
    
    best_score = -np.inf
    best_config = None
    
    for n_comp in n_components_range:
        for model_name in models.keys():
            if model_name in results[n_comp]['model_scores']:
                score = results[n_comp]['model_scores'][model_name]['mean_r2']
                if score > best_score:
                    best_score = score
                    best_config = (n_comp, model_name)
    
    best_n_comp, best_model_name = best_config
    print(f"Лучшая конфигурация:")
    print(f"  Number of components: {best_n_comp}")
    print(f"  Модель: {best_model_name}")
    print(f"  R² score: {best_score:.3f}")
    
    print("\n6. FINAL MODEL ON ALL DATA")
    print("-" * 40)
    
    best_pca = PCA(n_components=best_n_comp, random_state=42)
    X_pca_final = best_pca.fit_transform(X_scaled)
    
    best_model = models[best_model_name]
    best_model.fit(X_pca_final, y)
    
    y_pred = best_model.predict(X_pca_final)
    final_r2 = r2_score(y, y_pred)
    final_rmse = np.sqrt(mean_squared_error(y, y_pred))
    
    print(f"Финальная модель на всех данных:")
    print(f"  R²: {final_r2:.3f}")
    print(f"  RMSE: {final_rmse:.3f}")
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(y, y_pred, alpha=0.7, s=50)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
    plt.xlabel('True values')
    plt.ylabel('Predicted values')
    plt.title(f'Final model predictions\nR² = {final_r2:.3f}')
    
    plt.subplot(1, 2, 2)
    residuals = y - y_pred
    plt.scatter(y_pred, residuals, alpha=0.7, s=50)
    plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
    plt.xlabel('Predicted values')
    plt.ylabel('Residuals')
    plt.title('Residual analysis')
    
    plt.tight_layout()
    plt.show()
    
    print("\n7. PCA COMPONENT IMPORTANCE")
    print("-" * 40)
    
    component_importance = best_pca.explained_variance_ratio_
    
    plt.figure(figsize=(10, 6))
    plt.bar(range(1, best_n_comp + 1), component_importance)
    plt.xlabel('PCA component')
    plt.ylabel('Explained variance')
    plt.title(f'PCA component importance (total {best_n_comp} components)')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("Component importances:")
    for i, importance in enumerate(component_importance, 1):
        print(f"  Component {i}: {importance:.4f}")
    
    return {
        'scaler': scaler,
        'pca': best_pca,
        'model': best_model,
        'X_pca': X_pca_final,
        'results': results,
        'best_config': best_config,
        'final_metrics': {'r2': final_r2, 'rmse': final_rmse}
    }

In [ ]:
cleaned_df.columns

In [ ]:
fitdata= pd.concat([cleaned_df[["a", "b"]], cleaned_df.iloc[:, 5:]], axis = 1)

In [ ]:
cleaned_df

In [ ]:
fitdata

In [ ]:
fitdata.to_csv("../data/MLdata.csv")

In [ ]:
# result = complete_pca_analysis(pd.concat([cleaned_df[["a"]], cleaned_df.iloc[:, 5:]], axis = 1), "a", [3, 5, 8, 10, 12, 15, 17])

## PCA and parity workflows
Utilities for PCA sweeps, parity plots, and export of `MLdata.csv`.


## Run extended parity analysis
End-to-end parity helpers on the cleaned frame.


In [ ]:
result = complete_pca_analysis_with_parity(pd.concat([cleaned_df[["a"]], cleaned_df.iloc[:, 5:]], axis = 1), "a", [3, 5, 8, 10, 12, 15, 17, 20, 45, 50, 60])

In [ ]:
result = complete_pca_analysis_with_parity2(pd.concat([cleaned_df[["a"]], cleaned_df.iloc[:, 5:]], axis = 1), "a", [i for i in range(35, 60, 3)])